# Cognifyz Technologies - Data Analysis Internship
## Level 2 Analysis
**Candidate Name:** Jeevani Vedavendra  
**Reference ID:** CTI/A1/C421603  
**Domain:** Data Analysis  

---
### Overview of Level 2 Tasks:
1. **Task 1: Restaurant Ratings** - Analyze the distribution of aggregate ratings, determine the most common rating range, and calculate the average number of votes.
2. **Task 2: Cuisine Combination** - Identify the most common multi-cuisine combinations and determine if certain combinations tend to receive higher ratings.
3. **Task 3: Geographic Analysis** - Plot the locations of restaurants using latitude and longitude coordinates and identify patterns or clusters.
4. **Task 4: Restaurant Chains** - Identify restaurant chains in the dataset and analyze their ratings and popularity.


### Step 1: Import Libraries and Load Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'Segoe UI', 'font.size': 11})

df = pd.read_csv('../Dataset .csv')
print(f"Dataset Shape: {df.shape}")
df.head(3)


---
## Task 1: Restaurant Ratings
- Analyze the distribution of aggregate ratings and determine the most common rating range.
- Calculate the average number of votes received by restaurants.


In [ ]:
ratings = df['Aggregate rating']
mean_rating = ratings.mean()
median_rating = ratings.median()
avg_votes = df['Votes'].mean()

print(f"Mean Rating: {mean_rating:.2f}")
print(f"Median Rating: {median_rating:.2f}")
print(f"Average number of votes per restaurant: {avg_votes:.2f}")

# Categorical rating ranges
bins = [0, 1.9, 2.9, 3.9, 4.9, 5.0]
labels = ['0.0 - 1.9 (Poor/Unrated)', '2.0 - 2.9 (Average)', '3.0 - 3.9 (Good)', '4.0 - 4.9 (Very Good)', '5.0 (Perfect)']
df['Rating_Range'] = pd.cut(df['Aggregate rating'], bins=bins, labels=labels, include_lowest=True)
range_counts = df['Rating_Range'].value_counts().sort_index()

most_common_range = range_counts.idxmax()
print(f"\nMost common rating range: {most_common_range} ({range_counts[most_common_range]} restaurants, {range_counts[most_common_range]/len(df)*100:.2f}%)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Rating histogram & KDE
sns.histplot(df['Aggregate rating'], bins=20, kde=True, color='#2a9d8f', ax=axes[0])
axes[0].axvline(mean_rating, color='#e76f51', linestyle='--', linewidth=2, label=f'Mean: {mean_rating:.2f}')
axes[0].axvline(median_rating, color='#e9c46a', linestyle='-', linewidth=2, label=f'Median: {median_rating:.2f}')
axes[0].set_title("Distribution of Aggregate Ratings", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Aggregate Rating")
axes[0].legend()

# Rating range bar chart
range_df = range_counts.reset_index()
range_df.columns = ['Rating_Range', 'Count']
palette = ['#e63946' if r == most_common_range else '#457b9d' for r in range_df['Rating_Range']]
sns.barplot(data=range_df, x='Rating_Range', y='Count', palette=palette, hue='Rating_Range', legend=False, ax=axes[1])
axes[1].set_title("Most Common Rating Ranges (Highest Highlighted)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Rating Range")
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()


**Key Findings - Task 1:**
- The most common rating range is **3.0 - 3.9 (Good)** with **4,590 restaurants (48.06%)**.
- The overall average rating across all restaurants is **2.67**, and the average number of votes received is **156.91 votes**.


---
## Task 2: Cuisine Combinations
- Identify the most common combinations of cuisines in the dataset.
- Determine if certain cuisine combinations tend to have higher ratings.


In [ ]:
# Identify multi-cuisine combinations
cuisine_combos = df['Cuisines'].dropna()
combo_counts = cuisine_combos.value_counts()
multi_combos = combo_counts[combo_counts.index.str.contains(',')].head(10)

print("Top 5 Multi-Cuisine Combinations:")
for combo, count in multi_combos.head(5).items():
    print(f"- {combo}: {count} restaurants")

# Analyze ratings for combinations with >= 15 restaurants
combo_ratings = df.groupby('Cuisines').agg(
    Count=('Aggregate rating', 'count'),
    Avg_Rating=('Aggregate rating', 'mean')
).reset_index()

top_rated_combos = combo_ratings[combo_ratings['Count'] >= 15].sort_values(by='Avg_Rating', ascending=False).head(10)
display(top_rated_combos.head(5))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

most_freq_df = multi_combos.head(8).reset_index()
most_freq_df.columns = ['Combination', 'Count']
sns.barplot(data=most_freq_df, y='Combination', x='Count', palette='crest', hue='Combination', legend=False, ax=axes[0])
axes[0].set_title("Most Common Multi-Cuisine Combinations", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Number of Restaurants")

sns.barplot(data=top_rated_combos.head(8), y='Cuisines', x='Avg_Rating', palette='flare', hue='Cuisines', legend=False, ax=axes[1])
axes[1].set_title("Cuisine Combinations with Highest Average Ratings", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Average Rating (0 to 5)")
axes[1].set_xlim(0, 5)

plt.tight_layout()
plt.show()


**Key Findings - Task 2:**
- **North Indian, Chinese** is by far the most frequent pairing (511 restaurants), followed by **North Indian, Mughlai** (334 restaurants).
- Specialized combinations such as Japanese/Sushi (4.04), Thai (4.10), and Italian pairings achieve significantly higher average customer ratings than general fast-food pairings.


---
## Task 3: Geographic Analysis
- Plot the locations of restaurants on a map using longitude and latitude coordinates.
- Identify any patterns or clusters of restaurants in specific areas.


In [ ]:
# Filter out invalid (0,0) coordinates
geo_df = df[(df['Longitude'] != 0) & (df['Latitude'] != 0)].copy()
ncr_df = geo_df[geo_df['City'].isin(['New Delhi', 'Gurgaon', 'Noida', 'Faridabad', 'Ghaziabad'])]

print(f"Total valid coordinate entries: {len(geo_df)}")
print(f"Delhi NCR Cluster Count: {len(ncr_df)} ({len(ncr_df)/len(df)*100:.1f}%)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Global scatter
scatter1 = axes[0].scatter(geo_df['Longitude'], geo_df['Latitude'], c=geo_df['Aggregate rating'], cmap='Spectral_r', alpha=0.6, s=15)
fig.colorbar(scatter1, ax=axes[0], label='Aggregate Rating')
axes[0].set_title("Global Restaurant Coordinates", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

# Primary NCR cluster
scatter2 = axes[1].scatter(ncr_df['Longitude'], ncr_df['Latitude'], c=ncr_df['Aggregate rating'], cmap='Spectral_r', alpha=0.5, s=20)
fig.colorbar(scatter2, ax=axes[1], label='Aggregate Rating')
axes[1].set_title("Primary Cluster: Delhi NCR Region", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

plt.tight_layout()
plt.show()


**Key Findings - Task 3:**
- A massive geographic cluster is concentrated in the **Delhi NCR Region** (India), representing **79.0%** of the entire dataset.
- Secondary international clusters include the United States, UAE, Brazil, and South Africa.


---
## Task 4: Restaurant Chains
- Identify if there are any restaurant chains present in the dataset.
- Analyze the ratings and popularity of different restaurant chains.


In [ ]:
outlet_counts = df['Restaurant Name'].value_counts()
chains = outlet_counts[outlet_counts > 1]
total_chain_outlets = chains.sum()

print(f"Unique chain brands: {len(chains)}")
print(f"Total chain outlets: {total_chain_outlets} ({total_chain_outlets/len(df)*100:.2f}%)")

chain_df = df[df['Restaurant Name'].isin(chains.index)]
chain_stats = chain_df.groupby('Restaurant Name').agg(
    Outlets=('Aggregate rating', 'count'),
    Avg_Rating=('Aggregate rating', 'mean'),
    Total_Votes=('Votes', 'sum')
).reset_index()

top_chains_by_outlets = chain_stats.sort_values(by='Outlets', ascending=False).head(10)
display(top_chains_by_outlets)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=top_chains_by_outlets, y='Restaurant Name', x='Outlets', palette='viridis', hue='Restaurant Name', legend=False, ax=axes[0])
axes[0].set_title("Top 10 Restaurant Chains by Outlets", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Number of Outlets")

sns.barplot(data=top_chains_by_outlets, y='Restaurant Name', x='Avg_Rating', palette='mako', hue='Restaurant Name', legend=False, ax=axes[1])
axes[1].set_title("Average Rating of Top 10 Chains", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Average Rating (0 to 5)")
axes[1].set_xlim(0, 5)

plt.tight_layout()
plt.show()


**Key Findings - Task 4:**
- **Cafe Coffee Day** (83 outlets), **Domino's Pizza** (79 outlets), and **Subway** (63 outlets) are the largest chains by presence.
- Premium casual dining chains like **Chili's** (4.58) and **Barbeque Nation** (4.35) achieve the highest ratings among multi-location brands.
